## 📦 Install Dependencies

In [ ]:
!pip install plotly networkx numpy pandas ipywidgets

## 🧠 Generate Synthetic Mind Map

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import uuid

np.random.seed(42)

session_start = datetime(2025, 1, 15, 10, 0, 0)

# Topic names for each depth level
root_title = "Project Alpha"
depth1_titles = ["Research", "Design", "Backend", "Frontend"]
depth2_titles = [
    "Literature Review", "User Interviews", "Competitive Analysis", "Data Collection", "Trend Analysis",
    "Wireframes", "Color Palette", "Typography", "Iconography", "Prototyping",
    "Database Schema", "API Design", "Authentication", "Caching Layer", "Microservices",
    "Component Library", "State Management", "Routing", "Accessibility", "Performance Tuning"
]

nodes = []
time_offset = 0

# Root node (depth 0)
root_id = str(uuid.uuid4())
nodes.append({
    "id": root_id,
    "title": root_title,
    "focus_score": int(np.random.randint(60, 100)),
    "depth": 0,
    "parent_id": None,
    "created_at": session_start + timedelta(minutes=time_offset)
})
time_offset += 1

# Depth 1 nodes (4 children of root)
depth1_ids = []
for title in depth1_titles:
    node_id = str(uuid.uuid4())
    depth1_ids.append(node_id)
    nodes.append({
        "id": node_id,
        "title": title,
        "focus_score": int(np.random.randint(30, 95)),
        "depth": 1,
        "parent_id": root_id,
        "created_at": session_start + timedelta(minutes=time_offset)
    })
    time_offset += np.random.uniform(1, 3)

# Depth 2 nodes (5 grandchildren per depth-1 node = 20 total)
idx = 0
for parent_id in depth1_ids:
    for _ in range(5):
        node_id = str(uuid.uuid4())
        nodes.append({
            "id": node_id,
            "title": depth2_titles[idx],
            "focus_score": int(np.random.randint(10, 100)),
            "depth": 2,
            "parent_id": parent_id,
            "created_at": session_start + timedelta(minutes=time_offset)
        })
        time_offset += np.random.uniform(1, 2.5)
        idx += 1

df = pd.DataFrame(nodes)
print(f"Total nodes: {len(df)}")
df.head(10)

## 🔗 Build NetworkX Graph

In [ ]:
import pandas as pd
import networkx as nx

G = nx.DiGraph()

for _, row in df.iterrows():
    G.add_node(row["id"], title=row["title"], focus_score=row["focus_score"],
               depth=row["depth"], created_at=row["created_at"])

for _, row in df.iterrows():
    if pd.notna(row["parent_id"]):
        G.add_edge(row["parent_id"], row["id"])

print(f"Graph summary: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## 📐 3D Spring Layout

In [ ]:
pos_3d = nx.spring_layout(G, dim=3, seed=42, k=2)

node_positions = {}
for node_id, coords in pos_3d.items():
    node_positions[node_id] = {
        "x": coords[0],
        "y": coords[1],
        "z": coords[2]
    }

print(f"Computed 3D positions for {len(node_positions)} nodes")

## 🌐 3D Network Visualization

In [ ]:
import plotly.graph_objects as go

# Edge traces
edge_x, edge_y, edge_z = [], [], []
for src, dst in G.edges():
    x0, y0, z0 = pos_3d[src]
    x1, y1, z1 = pos_3d[dst]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]
    edge_z += [z0, z1, None]

edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode="lines",
    line=dict(color="gray", width=1),
    hoverinfo="none",
    showlegend=False
)

# Node attributes
node_x, node_y, node_z = [], [], []
node_colors, node_sizes, node_hover = [], [], []

for node_id in G.nodes():
    x, y, z = pos_3d[node_id]
    node_x.append(x)
    node_y.append(y)
    node_z.append(z)

    attrs = G.nodes[node_id]
    focus = attrs["focus_score"]
    title = attrs["title"]
    depth = attrs["depth"]
    num_children = len(list(G.successors(node_id)))

    node_colors.append(focus)
    node_sizes.append(max(8, min(25, 8 + num_children * 4)))
    node_hover.append(f"Title: {title}<br>Focus: {focus}<br>Depth: {depth}")

node_trace = go.Scatter3d(
    x=node_x, y=node_y, z=node_z,
    mode="markers+text",
    marker=dict(
        size=node_sizes,
        color=node_colors,
        colorscale=[[0, "red"], [0.5, "yellow"], [1, "green"]],
        cmin=0,
        cmax=100,
        colorbar=dict(title="Focus Score", thickness=15),
        line=dict(width=0.5, color="white")
    ),
    text=[G.nodes[n]["title"] for n in G.nodes()],
    textposition="top center",
    textfont=dict(size=8, color="white"),
    hovertext=node_hover,
    hoverinfo="text",
    showlegend=False
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title="Mind Map \u2014 3D Network Graph",
    template="plotly_dark",
    scene=dict(
        xaxis=dict(showbackground=False, visible=False),
        yaxis=dict(showbackground=False, visible=False),
        zaxis=dict(showbackground=False, visible=False),
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.5)
        )
    ),
    margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

## ⏱️ Focus Timeline — Session Playback

In [ ]:
df_sorted = df.sort_values("created_at").reset_index(drop=True)
df_sorted["time_offset_min"] = (df_sorted["created_at"] - df_sorted["created_at"].iloc[0]).dt.total_seconds() / 60

# Build frames for animation — each frame adds one more data point
frames = []
for i in range(1, len(df_sorted) + 1):
    frame_data = df_sorted.iloc[:i]
    frames.append(go.Frame(
        data=[go.Scatter(
            x=frame_data["time_offset_min"],
            y=frame_data["focus_score"],
            mode="lines+markers",
            line=dict(color="cyan", width=2),
            marker=dict(size=6, color="cyan"),
            text=frame_data["title"],
            hovertemplate="%{text}<br>Time: %{x:.1f} min<br>Focus: %{y}<extra></extra>"
        )],
        name=str(i)
    ))

# Initial trace (first point)
initial = df_sorted.iloc[:1]
fig_timeline = go.Figure(
    data=[go.Scatter(
        x=initial["time_offset_min"],
        y=initial["focus_score"],
        mode="lines+markers",
        line=dict(color="cyan", width=2),
        marker=dict(size=6, color="cyan"),
        text=initial["title"],
        hovertemplate="%{text}<br>Time: %{x:.1f} min<br>Focus: %{y}<extra></extra>"
    )],
    frames=frames
)

fig_timeline.update_layout(
    title="Focus Score Timeline \u2014 Session Playback",
    template="plotly_dark",
    xaxis_title="Time Offset (minutes)",
    yaxis_title="Focus Score",
    xaxis=dict(range=[0, df_sorted["time_offset_min"].max() + 1]),
    yaxis=dict(range=[0, 105]),
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        buttons=[
            dict(label="▶ Play",
                 method="animate",
                 args=[None, dict(frame=dict(duration=300, redraw=True), fromcurrent=True)]),
            dict(label="⏸ Pause",
                 method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])
        ]
    )],
    sliders=[dict(
        steps=[dict(args=[[str(i)], dict(frame=dict(duration=0, redraw=True), mode="immediate")],
                    method="animate", label=str(i)) for i in range(1, len(df_sorted) + 1)],
        currentvalue=dict(prefix="Node: "),
        pad=dict(t=50)
    )]
)
fig_timeline.show()

## 🔥 BCI Session Heatmap

In [ ]:
np.random.seed(99)

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
hours = list(range(9, 18))  # 9am to 5pm

# Simulate focus readings: 5 sessions x 9 hours
heatmap_data = np.zeros((len(days), len(hours)))
for d in range(len(days)):
    for h in range(len(hours)):
        # Simulate higher focus mid-morning, dip after lunch
        base = 50 + 20 * np.sin((hours[h] - 9) * np.pi / 9)
        noise = np.random.normal(0, 10)
        heatmap_data[d, h] = np.clip(base + noise + np.random.randint(-5, 15), 0, 100)

fig_heatmap = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=[f"{h}:00" for h in hours],
    y=days,
    colorscale="YlOrRd",
    zmin=0,
    zmax=100,
    colorbar=dict(title="Avg Focus Score")
))

fig_heatmap.update_layout(
    title="BCI Focus Heatmap \u2014 Time of Day vs Day of Week",
    template="plotly_dark",
    xaxis_title="Time of Day",
    yaxis_title="Day of Week",
    yaxis=dict(autorange="reversed")
)
fig_heatmap.show()

## 🎛️ Sandbox — Mind Map Explorer

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

print("\U0001f39b\ufe0f Sandbox \u2014 Mind Map Explorer")


def compute_layout(G, algorithm, seed=42):
    """Compute 3D layout using the selected algorithm."""
    if algorithm == "spring":
        return nx.spring_layout(G, dim=3, seed=seed, k=2)
    elif algorithm == "circular":
        pos_2d = nx.circular_layout(G)
        return {n: np.array([pos_2d[n][0], pos_2d[n][1], G.nodes[n]["depth"] * 0.5]) for n in G.nodes()}
    elif algorithm == "kamada_kawai":
        return nx.kamada_kawai_layout(G, dim=3)
    return nx.spring_layout(G, dim=3, seed=seed, k=2)


def get_node_colors(G, filtered_nodes, scheme, df):
    """Return color values and colorscale based on the selected scheme."""
    if scheme == "focus":
        colors = [G.nodes[n]["focus_score"] for n in filtered_nodes]
        colorscale = [[0, "red"], [0.5, "yellow"], [1, "green"]]
        cmin, cmax = 0, 100
        bar_title = "Focus Score"
    elif scheme == "depth":
        colors = [G.nodes[n]["depth"] for n in filtered_nodes]
        colorscale = [[0, "#e0f3ff"], [0.5, "#3b82f6"], [1, "#1e3a5f"]]
        cmin, cmax = 0, 2
        bar_title = "Depth"
    else:  # recency
        timestamps = [G.nodes[n]["created_at"] for n in filtered_nodes]
        t_min = min(timestamps)
        t_max = max(timestamps)
        t_range = (t_max - t_min).total_seconds() or 1
        colors = [(t - t_min).total_seconds() / t_range for t in timestamps]
        colorscale = [[0, "#f0f0f0"], [1, "#1a1a2e"]]
        cmin, cmax = 0, 1
        bar_title = "Recency"
    return colors, colorscale, cmin, cmax, bar_title


@interact(
    focus_threshold=widgets.IntSlider(min=0, max=100, value=30, description="Focus Min"),
    layout_algo=widgets.Dropdown(options=["spring", "circular", "kamada_kawai"], value="spring", description="Layout"),
    color_scheme=widgets.Dropdown(options=["focus", "depth", "recency"], value="focus", description="Color")
)
def explore_mind_map(focus_threshold, layout_algo, color_scheme):
    # Filter nodes by focus threshold
    filtered_nodes = [n for n in G.nodes() if G.nodes[n]["focus_score"] >= focus_threshold]
    if not filtered_nodes:
        print("No nodes above threshold.")
        return

    subG = G.subgraph(filtered_nodes).copy()
    pos = compute_layout(subG, layout_algo)

    # Edges
    ex, ey, ez = [], [], []
    for src, dst in subG.edges():
        x0, y0, z0 = pos[src]
        x1, y1, z1 = pos[dst]
        ex += [x0, x1, None]
        ey += [y0, y1, None]
        ez += [z0, z1, None]

    edge_tr = go.Scatter3d(x=ex, y=ey, z=ez, mode="lines",
                           line=dict(color="gray", width=1), hoverinfo="none", showlegend=False)

    # Nodes
    nx_arr, ny_arr, nz_arr = [], [], []
    hover_texts = []
    sizes = []
    for n in filtered_nodes:
        x, y, z = pos[n]
        nx_arr.append(x)
        ny_arr.append(y)
        nz_arr.append(z)
        attrs = subG.nodes[n]
        num_ch = len(list(subG.successors(n)))
        sizes.append(max(8, min(25, 8 + num_ch * 4)))
        hover_texts.append(f"Title: {attrs['title']}<br>Focus: {attrs['focus_score']}<br>Depth: {attrs['depth']}")

    colors, cscale, cmin, cmax, bar_title = get_node_colors(subG, filtered_nodes, color_scheme, df)

    node_tr = go.Scatter3d(
        x=nx_arr, y=ny_arr, z=nz_arr,
        mode="markers+text",
        marker=dict(size=sizes, color=colors, colorscale=cscale, cmin=cmin, cmax=cmax,
                    colorbar=dict(title=bar_title, thickness=15), line=dict(width=0.5, color="white")),
        text=[subG.nodes[n]["title"] for n in filtered_nodes],
        textposition="top center",
        textfont=dict(size=8, color="white"),
        hovertext=hover_texts,
        hoverinfo="text",
        showlegend=False
    )

    fig_sandbox = go.Figure(data=[edge_tr, node_tr])
    fig_sandbox.update_layout(
        title=f"Mind Map \u2014 {layout_algo} / {color_scheme} (threshold={focus_threshold})",
        template="plotly_dark",
        scene=dict(
            xaxis=dict(showbackground=False, visible=False),
            yaxis=dict(showbackground=False, visible=False),
            zaxis=dict(showbackground=False, visible=False),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
        ),
        margin=dict(l=0, r=0, t=40, b=0)
    )
    fig_sandbox.show()

## 💾 Export

In [ ]:
import os

os.makedirs("exports", exist_ok=True)
fig.write_html("exports/mind_map_3d.html")
print("\u2705 Exported 3D network graph to exports/mind_map_3d.html")